In [1]:
import sys
sys.path.insert(0, "..")  # repository root, so `import km5c` works
import numpy as np
import numpy.ma as ma
import os
import pandas as pd

from km5c.gridding import connect_atlantic
from km5c.masks import apply_atlantic_mask_2d
from km5c.stats import atlantic_summary_stats
from km5c.paths import TABLE_DIR
from km5c.models import load_cesm122, load_hadcm3, load_icesm12, load_icesm13
from km5c.proxies import comparison_tables

cesm122 = load_cesm122()
cesm122_Plio_d13C = cesm122.Plio_d13C
cesm122_Plio_depth = cesm122.Plio_depth
cesm122_Plio_lat = cesm122.Plio_lat
cesm122_Plio_lon = cesm122.Plio_lon
cesm122_preind_d13C = cesm122.preind_d13C
cesm122_preind_depth = cesm122.preind_depth
cesm122_preind_lat = cesm122.preind_lat
cesm122_preind_lon = cesm122.preind_lon

hadcm3 = load_hadcm3()
hadcm3_Plio_d18O = hadcm3.Plio_d18O
hadcm3_Plio_depth = hadcm3.Plio_depth
hadcm3_Plio_lat = hadcm3.Plio_lat
hadcm3_Plio_lon = hadcm3.Plio_lon
hadcm3_Plio_temp = hadcm3.Plio_temp
hadcm3_preind_d18O = hadcm3.preind_d18O
hadcm3_preind_depth = hadcm3.preind_depth
hadcm3_preind_lat = hadcm3.preind_lat
hadcm3_preind_lon = hadcm3.preind_lon
hadcm3_preind_temp = hadcm3.preind_temp

icesm12 = load_icesm12()
Plio_d18O = icesm12.Plio_d18O
Plio_depth = icesm12.Plio_depth
Plio_lat = icesm12.Plio_lat
Plio_lon = icesm12.Plio_lon
Plio_temp = icesm12.Plio_temp
preind_d18O = icesm12.preind_d18O
preind_depth = icesm12.preind_depth
preind_lat = icesm12.preind_lat
preind_lon = icesm12.preind_lon
preind_temp = icesm12.preind_temp

icesm13 = load_icesm13()
icesm13_Plio_d18O = icesm13.Plio_d18O
icesm13_Plio_depth = icesm13.Plio_depth
icesm13_Plio_lat = icesm13.Plio_lat
icesm13_Plio_lon = icesm13.Plio_lon
icesm13_Plio_temp = icesm13.Plio_temp
icesm13_preind_d18O = icesm13.preind_d18O
icesm13_preind_depth = icesm13.preind_depth
icesm13_preind_lat = icesm13.preind_lat
icesm13_preind_lon = icesm13.preind_lon
icesm13_preind_temp = icesm13.preind_temp

proxy = comparison_tables()
PLIOMIP2_d13C_comparison_table = proxy.PLIOMIP2_d13C_comparison_table
PLIOMIP2_d18O_comparison_table = proxy.PLIOMIP2_d18O_comparison_table

(1, 60, 180, 360)
(60, 180, 360)


# Atlantic (NADW) summary statistics, below 200 m

Volume-weighted statistics of the model output inside the Atlantic polygon used for the lat-depth figures
(`apply_atlantic_mask_3d`), excluding the top 200 m, for the pre-industrial and Pliocene (KM5c) runs.
Statistics are computed on each model's native grid (not the KNN-interpolated, z-scored sections).

- d18O calcite = -0.224 * T + 3.26 + d18O seawater (same conversion as `temperatureadjustedplot`)
- Weights = cell area x layer thickness; n_cells counts wet cells.

In [2]:
MIN_DEPTH = 200.0  # m, drop the top 200 m

def d18O_calcite(d18O_sw, temp):
    return -0.224 * temp + 3.26 + d18O_sw

cases = [
    # (variable, model, period, field, lat, lon, depth, depth_is_top)
    ("d18O seawater", "iCESM1.2-ITPCAS", "Pre-industrial", preind_d18O, preind_lat, preind_lon, preind_depth, False),
    ("d18O seawater", "iCESM1.2-ITPCAS", "Pliocene", Plio_d18O, Plio_lat, Plio_lon, Plio_depth, False),
    ("d18O seawater", "iCESM1.3", "Pre-industrial", icesm13_preind_d18O, icesm13_preind_lat, icesm13_preind_lon, icesm13_preind_depth, False),
    ("d18O seawater", "iCESM1.3", "Pliocene", icesm13_Plio_d18O, icesm13_Plio_lat, icesm13_Plio_lon, icesm13_Plio_depth, False),
    ("d18O seawater", "HadCM3", "Pre-industrial", hadcm3_preind_d18O, hadcm3_preind_lat, hadcm3_preind_lon, hadcm3_preind_depth, False),
    ("d18O seawater", "HadCM3", "Pliocene", hadcm3_Plio_d18O, hadcm3_Plio_lat, hadcm3_Plio_lon, hadcm3_Plio_depth, False),

    ("d18O calcite", "iCESM1.2-ITPCAS", "Pre-industrial", d18O_calcite(preind_d18O, preind_temp), preind_lat, preind_lon, preind_depth, False),
    ("d18O calcite", "iCESM1.2-ITPCAS", "Pliocene", d18O_calcite(Plio_d18O, Plio_temp), Plio_lat, Plio_lon, Plio_depth, False),
    ("d18O calcite", "iCESM1.3", "Pre-industrial", d18O_calcite(icesm13_preind_d18O, icesm13_preind_temp), icesm13_preind_lat, icesm13_preind_lon, icesm13_preind_depth, False),
    ("d18O calcite", "iCESM1.3", "Pliocene", d18O_calcite(icesm13_Plio_d18O, icesm13_Plio_temp), icesm13_Plio_lat, icesm13_Plio_lon, icesm13_Plio_depth, False),
    ("d18O calcite", "HadCM3", "Pre-industrial", d18O_calcite(hadcm3_preind_d18O, hadcm3_preind_temp), hadcm3_preind_lat, hadcm3_preind_lon, hadcm3_preind_depth, False),
    ("d18O calcite", "HadCM3", "Pliocene", d18O_calcite(hadcm3_Plio_d18O, hadcm3_Plio_temp), hadcm3_Plio_lat, hadcm3_Plio_lon, hadcm3_Plio_depth, False),

    # CESM1.2.2.1 d13C uses z_w (layer tops), as in the d13C figures
    ("d13C DIC", "CESM1.2.2.1", "Pre-industrial", cesm122_preind_d13C, cesm122_preind_lat, cesm122_preind_lon, cesm122_preind_depth, True),
    ("d13C DIC", "CESM1.2.2.1", "Pliocene", cesm122_Plio_d13C, cesm122_Plio_lat, cesm122_Plio_lon, cesm122_Plio_depth, True),
]

rows = []
for variable, model, period, field, lat, lon, depth, depth_is_top in cases:
    stats = atlantic_summary_stats(field, lat, lon, depth, min_depth=MIN_DEPTH, depth_is_top=depth_is_top)
    rows.append({"variable": variable, "model": model, "period": period, **stats})

summary_stats = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.3f}".format)
summary_stats

,variable,model,period,n_cells,volume_km3,mean,std,median,p10,p90,min,max
0,d18O seawater,iCESM1.2-ITPCAS,Pre-industrial,280018,288303353.839,0.067,0.149,0.026,-0.058,0.235,-0.840,0.968
1,d18O seawater,iCESM1.2-ITPCAS,Pliocene,279439,280568973.854,0.159,0.162,0.184,-0.060,0.317,-0.987,0.848
2,d18O seawater,iCESM1.3,Pre-industrial,650872,307240457.759,0.189,0.160,0.212,-0.025,0.307,-1.108,2.406
3,d18O seawater,iCESM1.3,Pliocene,292813,299899393.156,0.108,0.163,0.105,-0.093,0.276,-0.556,1.135
4,d18O seawater,HadCM3,Pre-industrial,55864,334757558.813,0.074,0.105,0.092,-0.083,0.178,-0.286,0.559
5,d18O seawater,HadCM3,Pliocene,57660,334465911.016,-0.177,0.122,-0.158,-0.367,-0.051,-0.542,0.297
6,d18O calcite,iCESM1.2-ITPCAS,Pre-industrial,280018,288303353.839,2.250,0.558,2.389,1.483,2.821,-0.865,3.403
7,d18O calcite,iCESM1.2-ITPCAS,Pliocene,279439,280568973.854,1.468,0.492,1.588,0.806,1.958,-1.095,2.831
8,d18O calcite,iCESM1.3,Pre-industrial,650872,307240457.759,2.479,0.593,2.643,1.647,3.059,-0.872,3.843
9,d18O calcite,iCESM1.3,Pliocene,292813,299899393.156,1.771,0.628,1.873,0.915,2.417,-1.310,3.198


In [3]:
# Pliocene minus pre-industrial for each statistic
stat_cols = ["mean", "std", "median", "p10", "p90"]
wide = summary_stats.pivot_table(index=["variable", "model"], columns="period", values=stat_cols, sort=False)
anomaly = (wide.xs("Pliocene", axis=1, level="period") - wide.xs("Pre-industrial", axis=1, level="period"))[stat_cols]
anomaly.columns = [f"{c} (Plio - PI)" for c in anomaly.columns]
anomaly

mean (Plio - PI)  std (Plio - PI)  \
variable      model                                                
d18O seawater iCESM1.2-ITPCAS             0.092            0.013   
              iCESM1.3                   -0.082            0.003   
              HadCM3                     -0.251            0.017   
d18O calcite  iCESM1.2-ITPCAS            -0.782           -0.066   
              iCESM1.3                   -0.708            0.035   
              HadCM3                     -1.087           -0.187   
d13C DIC      CESM1.2.2.1                -0.735            0.257   

                               median (Plio - PI)  p10 (Plio - PI)  \
variable      model                                                  
d18O seawater iCESM1.2-ITPCAS               0.158           -0.002   
              iCESM1.3                     -0.107           -0.068   
              HadCM3                       -0.250           -0.284   
d18O calcite  iCESM1.2-ITPCAS              -0.801           -0.677   
              iCESM1.3                     -0.770           -0.732   
              HadCM3                       -1.245           -0.658   
d13C DIC      CESM1.2.2.1                  -0.782           -1.146   

                               p90 (Plio - PI)  
variable      model                             
d18O seawater iCESM1.2-ITPCAS            0.082  
              iCESM1.3                  -0.031  
              HadCM3                    -0.229  
d18O calcite  iCESM1.2-ITPCAS           -0.863  
              iCESM1.3                  -0.642  
              HadCM3                    -1.246  
d13C DIC      CESM1.2.2.1               -0.280

## Proxy data in the same region

OC3 (pre-industrial) and PlioVAR KM5c (Pliocene) benthic values from the comparison tables used in the figures,
restricted to sites inside the same Atlantic polygon and deeper than 200 m. Mean and s.d. are across sites (unweighted).
The model equivalents are **d18O calcite** and **d13C DIC**.

In [4]:
proxy_sources = [
    # (variable, table, value column, period)
    ("d18O calcite", PLIOMIP2_d18O_comparison_table, "OC3 d18O", "Pre-industrial"),
    ("d18O calcite", PLIOMIP2_d18O_comparison_table, "PLIOMIP2 d18O", "Pliocene"),
    ("d13C DIC", PLIOMIP2_d13C_comparison_table, "OC3 d13C", "Pre-industrial"),
    ("d13C DIC", PLIOMIP2_d13C_comparison_table, "PLIOMIP2 d13C", "Pliocene"),
]

def in_atlantic_below(table, min_depth=MIN_DEPTH):
    lat = pd.to_numeric(table["Latitude"]).to_numpy()
    lon = connect_atlantic(pd.to_numeric(table["Longitude"]).to_numpy())
    depth = pd.to_numeric(table["Water depth (m)"]).to_numpy()
    # Same polygon as the model fields (apply_atlantic_mask_2d / _3d)
    inside = ~ma.getmaskarray(apply_atlantic_mask_2d(np.zeros(lat.shape), lat, lon))
    return inside & (depth >= min_depth)

proxy_rows = []
for variable, table, column, period in proxy_sources:
    table = table[table["Site name"] != "Average"]
    values = pd.to_numeric(table[column], errors="coerce")[in_atlantic_below(table)].dropna()
    proxy_rows.append({"variable": variable, "model": "Proxies (OC3 / PlioVAR)", "period": period,
                       "n_sites": len(values), "mean": values.mean(), "std": values.std(ddof=1),
                       "median": values.median(), "min": values.min(), "max": values.max()})
proxy_stats = pd.DataFrame(proxy_rows)

excluded = PLIOMIP2_d18O_comparison_table.loc[~in_atlantic_below(PLIOMIP2_d18O_comparison_table), "Site name"].tolist()
print("Sites excluded (outside polygon or < 200 m):", excluded)
proxy_stats

Sites excluded (outside polygon or < 200 m): [999]


,variable,model,period,n_sites,mean,std,median,min,max
0,d18O calcite,Proxies (OC3 / PlioVAR),Pre-industrial,18,2.629,0.399,2.584,1.775,3.456
1,d18O calcite,Proxies (OC3 / PlioVAR),Pliocene,22,2.453,0.249,2.426,2.066,2.908
2,d13C DIC,Proxies (OC3 / PlioVAR),Pre-industrial,17,0.760,0.300,0.850,0.040,1.102
3,d13C DIC,Proxies (OC3 / PlioVAR),Pliocene,19,0.743,0.355,0.845,-0.280,1.150


In [5]:
# Side-by-side: model volume-weighted stats vs proxy site stats
comparison = pd.concat([summary_stats[summary_stats["variable"] != "d18O seawater"], proxy_stats], ignore_index=True)
comparison = comparison[["variable", "period", "model", "mean", "std", "median", "min", "max", "n_cells", "n_sites"]]
comparison = comparison.sort_values(["variable", "period", "model"], ignore_index=True)

proxy_anomaly = proxy_stats.set_index(["variable", "period"])["mean"].unstack("period")
proxy_anomaly["mean (Plio - PI)"] = proxy_anomaly["Pliocene"] - proxy_anomaly["Pre-industrial"]
display(comparison)
proxy_anomaly

,variable,period,model,mean,std,median,min,max,n_cells,n_sites
0,d13C DIC,Pliocene,CESM1.2.2.1,0.125,0.636,0.032,-1.542,1.834,268695.000,NaN
1,d13C DIC,Pliocene,Proxies (OC3 / PlioVAR),0.743,0.355,0.845,-0.280,1.150,NaN,19.000
2,d13C DIC,Pre-industrial,CESM1.2.2.1,0.860,0.378,0.814,-0.610,2.445,268695.000,NaN
3,d13C DIC,Pre-industrial,Proxies (OC3 / PlioVAR),0.760,0.300,0.850,0.040,1.102,NaN,17.000
4,d18O calcite,Pliocene,HadCM3,1.156,0.527,1.242,-2.691,3.044,56808.000,NaN
5,d18O calcite,Pliocene,Proxies (OC3 / PlioVAR),2.453,0.249,2.426,2.066,2.908,NaN,22.000
6,d18O calcite,Pliocene,iCESM1.2-ITPCAS,1.468,0.492,1.588,-1.095,2.831,279439.000,NaN
7,d18O calcite,Pliocene,iCESM1.3,1.771,0.628,1.873,-1.310,3.198,292813.000,NaN
8,d18O calcite,Pre-industrial,HadCM3,2.243,0.714,2.488,-1.740,3.555,55864.000,NaN
9,d18O calcite,Pre-industrial,Proxies (OC3 / PlioVAR),2.629,0.399,2.584,1.775,3.456,NaN,18.000


period,Pliocene,Pre-industrial,mean (Plio - PI)
variable,,,
d13C DIC,0.743,0.760,-0.017
d18O calcite,2.453,2.629,-0.176


In [6]:
os.makedirs(TABLE_DIR, exist_ok=True)
summary_stats.to_csv(TABLE_DIR / "atlantic_below200m_summary_stats.csv", index=False)
anomaly.to_csv(TABLE_DIR / "atlantic_below200m_plio_minus_pi.csv")
comparison.to_csv(TABLE_DIR / "atlantic_below200m_model_vs_proxy.csv", index=False)